# ✦ laboratorio.ia · entrena tu propio modelo

Este cuaderno convierte tus conversaciones de **laboratorio.ia** en un modelo propio (adaptador **LoRA**) y lo sube a tu Cloudflare para usarlo en la app.

1. En laboratorio.ia: **Ajustes → Entrenar tu propio modelo** → descarga `SFT` (o `DPO`).
2. Aquí: **Entorno de ejecución → Cambiar tipo → GPU T4** (gratis) y ejecuta las celdas en orden (▶).
3. Al final, pega el nombre del modelo en laboratorio.ia → *Usa tu modelo aquí*.

*SFT* enseña a responder como tus mejores respuestas. *DPO* enseña a preferir la respuesta buena frente a la peor (regeneradas, combinaciones, tu ★). Lo habitual: primero SFT; DPO cuando tengas 30+ pares.

In [ ]:
#@title 1 · Ajustes { display-mode: "form" }
MODO = "sft"  #@param ["sft", "dpo"]
BASE = "llama-3.2-3b"  #@param ["llama-3.2-3b", "mistral-7b", "gemma-7b", "gemma-2b"]
NOMBRE = "mi-laboratorio-v1"  #@param {type:"string"}
EPOCAS = 2  #@param {type:"integer"}
MAX_TOKENS = 4096  #@param {type:"integer"}

BASES = {
    "llama-3.2-3b": dict(hf="unsloth/Llama-3.2-3B-Instruct", cf="@cf/meta/llama-3.2-3b-instruct", tipo="llama", sistema=True),
    "mistral-7b":   dict(hf="unsloth/mistral-7b-instruct-v0.2", cf="@cf/mistral/mistral-7b-instruct-v0.2-lora", tipo="mistral", sistema=False),
    "gemma-7b":     dict(hf="unsloth/gemma-7b-it", cf="@cf/google/gemma-7b-it-lora", tipo="gemma", sistema=False),
    "gemma-2b":     dict(hf="unsloth/gemma-2b-it", cf="@cf/google/gemma-2b-it-lora", tipo="gemma", sistema=False),
}
B = BASES[BASE]
# Workers AI acepta adaptadores de rango <= 32; su ejemplo usa q_proj y v_proj (lo más compatible).
RANGO, MODULOS = 8, ["q_proj", "v_proj"]
print("Base:", B["hf"], "→ Workers AI:", B["cf"])

In [ ]:
#@title 2 · Instalar Unsloth (2-3 min)
!pip install -q unsloth

In [ ]:
#@title 3 · Sube el archivo .jsonl que descargaste de laboratorio.ia
from google.colab import files
subido = files.upload()
ARCHIVO = next(iter(subido))
import json
filas = [json.loads(l) for l in open(ARCHIVO, encoding="utf-8") if l.strip()]
print(len(filas), "ejemplos en", ARCHIVO)
if MODO == "dpo" and "chosen" not in filas[0]: raise SystemExit("Este archivo es SFT: elige MODO = sft o sube el de DPO")
if MODO == "sft" and "messages" not in filas[0]: raise SystemExit("Este archivo es DPO: elige MODO = dpo o sube el de SFT")

In [ ]:
#@title 4 · Cargar el modelo base y preparar el adaptador LoRA
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(B["hf"], max_seq_length=MAX_TOKENS, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(model, r=RANGO, lora_alpha=RANGO * 2, lora_dropout=0, target_modules=MODULOS,
                                         bias="none", use_gradient_checkpointing="unsloth", random_state=7)

def plano(mensajes):
    """Mistral y Gemma no tienen rol «system»: se une a la primera pregunta."""
    if B["sistema"] or not mensajes or mensajes[0]["role"] != "system": return mensajes
    resto = [dict(m) for m in mensajes[1:]]
    if resto and resto[0]["role"] == "user": resto[0]["content"] = mensajes[0]["content"] + "\n\n" + resto[0]["content"]
    return resto

def recortar(texto, n=MAX_TOKENS * 3):  # ~3 caracteres por token: evita ejemplos que no caben
    return texto if len(texto) <= n else texto[-n:]

In [ ]:
#@title 5 · Entrenar (10-40 min según los datos)
import inspect
from datasets import Dataset

def con_firma(clase, **kw):
    """Las versiones de TRL cambian nombres de parámetros; se pasan solo los que existen."""
    p = inspect.signature(clase.__init__).parameters
    if "max_seq_length" in kw and "max_seq_length" not in p: kw["max_length"] = kw.pop("max_seq_length")
    return {k: v for k, v in kw.items() if k in p}

comun = dict(per_device_train_batch_size=2, gradient_accumulation_steps=4, num_train_epochs=EPOCAS, warmup_steps=5,
             logging_steps=5, optim="adamw_8bit", lr_scheduler_type="linear", seed=7, output_dir="salida", report_to="none",
             fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported())

if MODO == "sft":
    from trl import SFTTrainer, SFTConfig
    textos = [recortar(tokenizer.apply_chat_template(plano(f["messages"]), tokenize=False)) for f in filas]
    datos = Dataset.from_dict({"text": textos})
    args = SFTConfig(**con_firma(SFTConfig, learning_rate=2e-4, dataset_text_field="text", max_seq_length=MAX_TOKENS, **comun))
    kw = con_firma(SFTTrainer, model=model, train_dataset=datos, args=args, tokenizer=tokenizer, processing_class=tokenizer)
    entrenador = SFTTrainer(**kw)
else:
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
    from trl import DPOTrainer, DPOConfig
    def fila(f):
        prompt = tokenizer.apply_chat_template(plano(f["prompt"]), tokenize=False, add_generation_prompt=True)
        return {"prompt": recortar(prompt), "chosen": f["chosen"][0]["content"] + tokenizer.eos_token, "rejected": f["rejected"][0]["content"] + tokenizer.eos_token}
    datos = Dataset.from_list([fila(f) for f in filas])
    args = DPOConfig(**con_firma(DPOConfig, learning_rate=5e-6, beta=0.1, max_length=MAX_TOKENS, max_prompt_length=MAX_TOKENS // 2, **comun))
    kw = con_firma(DPOTrainer, model=model, ref_model=None, train_dataset=datos, args=args, tokenizer=tokenizer, processing_class=tokenizer)
    entrenador = DPOTrainer(**kw)

entrenador.train()

In [ ]:
#@title 6 · Pruébalo
FastLanguageModel.for_inference(model)
pregunta = "¿Qué evidencia hay sobre la metformina en el envejecimiento?"  #@param {type:"string"}
mensajes = plano([{"role": "system", "content": "Eres laboratorio.ia, un asistente de investigación científica."}, {"role": "user", "content": pregunta}])
ids = tokenizer.apply_chat_template(mensajes, add_generation_prompt=True, return_tensors="pt").to("cuda")
salida = model.generate(input_ids=ids, max_new_tokens=600, temperature=0.4, do_sample=True)
print(tokenizer.decode(salida[0][ids.shape[1]:], skip_special_tokens=True))

In [ ]:
#@title 7 · Guardar el adaptador en el formato de Workers AI
import os, shutil
model.save_pretrained("adaptador")
cfg = json.load(open("adaptador/adapter_config.json"))
cfg["model_type"] = B["tipo"]  # Workers AI lo exige: llama, mistral o gemma
json.dump(cfg, open("adaptador/adapter_config.json", "w"), indent=2)
tam = os.path.getsize("adaptador/adapter_model.safetensors") / 1e6
print(f"adapter_model.safetensors: {tam:.1f} MB (máx. 300 MB)")
shutil.make_archive(NOMBRE, "zip", "adaptador")
files.download(NOMBRE + ".zip")  # copia de seguridad: sirve también para Ollama / LM Studio con el modelo base

In [ ]:
#@title 8 · Subir a tu Cloudflare (Workers AI)
# Necesitas el ID de cuenta (dash.cloudflare.com → barra lateral) y un token con permiso «Workers AI: Edit»
# (dash.cloudflare.com/profile/api-tokens → Crear token → plantilla «Workers AI»).
import requests, getpass
CUENTA = input("ID de cuenta de Cloudflare: ").strip()
TOKEN = getpass.getpass("Token de API (Workers AI: Edit): ").strip()
api = f"https://api.cloudflare.com/client/v4/accounts/{CUENTA}/ai/finetunes"
h = {"Authorization": f"Bearer {TOKEN}"}
r = requests.post(api, headers=h, json={"model": B["cf"], "name": NOMBRE, "description": f"laboratorio.ia · {MODO} · {len(filas)} ejemplos"}).json()
if not r.get("success"): raise SystemExit(f"Cloudflare: {r.get('errors')}")
fid = r["result"]["id"]
for nombre in ["adapter_config.json", "adapter_model.safetensors"]:
    with open(f"adaptador/{nombre}", "rb") as fh:
        s = requests.post(f"{api}/{fid}/finetune-assets/", headers=h, files={"file": (nombre, fh)}, data={"file_name": nombre}).json()
    print(nombre, "✓" if s.get("success") else s.get("errors"))
print(f"\nListo. En laboratorio.ia → Ajustes → Entrenar tu propio modelo → «Usa tu modelo aquí»:\n  modelo base: {B['cf']}\n  nombre: {NOMBRE}")